In [ ]:
%cd /content

/content


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir /content/preprocessing
!cp -r /content/drive/MyDrive/NewDataset/cloth-mask.zip /content/preprocessing
!cp -r /content/drive/MyDrive/NewDataset/cloths.zip /content/preprocessing
!cp -r /content/drive/MyDrive/NewDataset/image-densepose.zip /content/preprocessing
!cp -r /content/drive/MyDrive/NewDataset/image-parse-v3.zip /content/preprocessing
!cp -r /content/drive/MyDrive/NewDataset/movenet_img.zip /content/preprocessing
!cp -r /content/drive/MyDrive/NewDataset/movenet_json.zip /content/preprocessing
!cp -r /content/drive/MyDrive/NewDataset/image.zip /content/preprocessing
!cp -r /content/drive/MyDrive/NewDataset/image-parse-agnostic-v3.2.zip /content/preprocessing

mkdir: cannot create directory ‘/content/preprocessing’: File exists


In [ ]:
!unzip /content/preprocessing/cloth-mask.zip -d /content/preprocessing
!unzip /content/preprocessing/cloths.zip -d /content/preprocessing
!unzip /content/preprocessing/image-densepose.zip -d /content/preprocessing
!unzip /content/preprocessing/image-parse-v3.zip -d /content/preprocessing
!unzip /content/preprocessing/movenet_img.zip -d /content/preprocessing
!unzip /content/preprocessing/movenet_json.zip -d /content/preprocessing
!unzip /content/preprocessing/image.zip -d /content/preprocessing
!unzip /content/preprocessing/image-parse-agnostic-v3.2.zip -d /content/preprocessing

Streaming output truncated to the last 5000 lines.
  inflating: /content/preprocessing/image-parse-v3/02537_00.png  
  inflating: /content/preprocessing/image-parse-v3/02538_00.png  
  inflating: /content/preprocessing/image-parse-v3/02539_00.png  
  inflating: /content/preprocessing/image-parse-v3/02540_00.png  
  inflating: /content/preprocessing/image-parse-v3/02541_00.png  
  inflating: /content/preprocessing/image-parse-v3/02542_00.png  
  inflating: /content/preprocessing/image-parse-v3/02543_00.png  
  inflating: /content/preprocessing/image-parse-v3/02544_00.png  
  inflating: /content/preprocessing/image-parse-v3/02545_00.png  
  inflating: /content/preprocessing/image-parse-v3/02565_00.png  
  inflating: /content/preprocessing/image-parse-v3/02579_00.png  
  inflating: /content/preprocessing/image-parse-v3/02582_00.png  
  inflating: /content/preprocessing/image-parse-v3/02583_00.png  
  inflating: /content/preprocessing/image-parse-v3/02584_00.png  
  inflating: /content/pre

In [ ]:
!rm -rf /content/preprocessing/cloth-mask.zip
!rm -rf /content/preprocessing/cloths.zip
!rm -rf /content/preprocessing/image-densepose.zip
!rm -rf /content/preprocessing/image-parse-v3.zip
!rm -rf /content/preprocessing/movenet_img.zip
!rm -rf /content/preprocessing/movenet_json.zip
!rm -rf /content/preprocessing/image.zip
!rm -rf /content/preprocessing/image-parse-agnostic-v3.2.zip

In [ ]:
import json
from os import path as osp
import os
import numpy as np
from PIL import Image, ImageDraw
from tqdm import tqdm

def get_img_agnostic(img, parse, pose_data):
    parse_array = np.array(parse)
    parse_head = ((parse_array == 4).astype(np.float32) +
                  (parse_array == 13).astype(np.float32))
    parse_lower = ((parse_array == 9).astype(np.float32) +
                   (parse_array == 12).astype(np.float32) +
                   (parse_array == 16).astype(np.float32) +
                   (parse_array == 17).astype(np.float32) +
                   (parse_array == 18).astype(np.float32) +
                   (parse_array == 19).astype(np.float32))

    agnostic = img.copy()
    agnostic_draw = ImageDraw.Draw(agnostic)

    length_a = np.linalg.norm(pose_data[5] - pose_data[2])
    length_b = np.linalg.norm(pose_data[12] - pose_data[9])
    point = (pose_data[9] + pose_data[12]) / 2
    pose_data[9] = point + (pose_data[9] - point) / length_b * length_a
    pose_data[12] = point + (pose_data[12] - point) / length_b * length_a
    r = int(length_a / 16) + 1

    # mask arms
    agnostic_draw.line([tuple(pose_data[i]) for i in [2, 5]], 'gray', width=r*10)
    for i in [2, 5]:
        pointx, pointy = pose_data[i]
        agnostic_draw.ellipse((pointx-r*5, pointy-r*5, pointx+r*5, pointy+r*5), 'gray', 'gray')
    for i in [3, 4, 6, 7]:
        if (pose_data[i - 1, 0] == 0.0 and pose_data[i - 1, 1] == 0.0) or (pose_data[i, 0] == 0.0 and pose_data[i, 1] == 0.0):
            continue
        agnostic_draw.line([tuple(pose_data[j]) for j in [i - 1, i]], 'gray', width=r*10)
        pointx, pointy = pose_data[i]
        agnostic_draw.ellipse((pointx-r*5, pointy-r*5, pointx+r*5, pointy+r*5), 'gray', 'gray')

    # mask torso
    for i in [9, 12]:
        pointx, pointy = pose_data[i]
        agnostic_draw.ellipse((pointx-r*3, pointy-r*6, pointx+r*3, pointy+r*6), 'gray', 'gray')
    agnostic_draw.line([tuple(pose_data[i]) for i in [2, 9]], 'gray', width=r*6)
    agnostic_draw.line([tuple(pose_data[i]) for i in [5, 12]], 'gray', width=r*6)
    agnostic_draw.line([tuple(pose_data[i]) for i in [9, 12]], 'gray', width=r*12)
    agnostic_draw.polygon([tuple(pose_data[i]) for i in [2, 5, 12, 9]], 'gray', 'gray')

    # mask neck
    pointx, pointy = pose_data[1]
    agnostic_draw.rectangle((pointx-r*7, pointy-r*7, pointx+r*7, pointy+r*7), 'gray', 'gray')

    # Ensure mask is resized to match image size
    mask_head = Image.fromarray(np.uint8(parse_head * 255), 'L').resize(img.size)
    mask_lower = Image.fromarray(np.uint8(parse_lower * 255), 'L').resize(img.size)

    # Apply masks
    agnostic.paste(img, None, mask_head)
    agnostic.paste(img, None, mask_lower)

    return agnostic

if __name__ == "__main__":
    data_path = '/content/preprocessing'
    output_path = './test/agnostic-v3.2'

    os.makedirs(output_path, exist_ok=True)

    for im_name in tqdm(os.listdir(osp.join(data_path, 'image'))):

        # load pose data
        pose_name = im_name.replace('.jpg', '_keypoints.json')

        try:
            with open(osp.join(data_path, 'movenet_json', pose_name), 'r') as f:
                pose_label = json.load(f)
                pose_data = pose_label['people'][0]['pose_keypoints_2d']
                pose_data = np.array(pose_data)
                pose_data = pose_data.reshape((-1, 3))[:, :2]
        except (IndexError, FileNotFoundError, KeyError):
            print(f"Skipping: {pose_name}")
            continue

        # load image and parsing label
        im = Image.open(osp.join(data_path, 'image', im_name))
        label_name = im_name.replace('.jpg', '.png')
        im_label = Image.open(osp.join(data_path, 'image-parse-v3', label_name))

        agnostic = get_img_agnostic(im, im_label, pose_data)

        agnostic.convert("RGB").save(osp.join(output_path, im_name))



100%|██████████| 1177/1177 [01:46<00:00, 11.08it/s]


In [ ]:
!zip -r agnostic-v3.2.zip /content/test/agnostic-v3.2
!cp agnostic-v3.2.zip /content/drive/MyDrive/NewDataset/

  adding: content/test/agnostic-v3.2/ (stored 0%)
  adding: content/test/agnostic-v3.2/00283_00.jpg (deflated 10%)
  adding: content/test/agnostic-v3.2/00499_00.jpg (deflated 11%)
  adding: content/test/agnostic-v3.2/02913_00.jpg (deflated 14%)
  adding: content/test/agnostic-v3.2/01595_00.jpg (deflated 11%)
  adding: content/test/agnostic-v3.2/01012_00.jpg (deflated 23%)
  adding: content/test/agnostic-v3.2/02199_00.jpg (deflated 7%)
  adding: content/test/agnostic-v3.2/02381_00.jpg (deflated 15%)
  adding: content/test/agnostic-v3.2/03030_00.jpg (deflated 13%)
  adding: content/test/agnostic-v3.2/02979_00.jpg (deflated 14%)
  adding: content/test/agnostic-v3.2/02665_00.jpg (deflated 1%)
  adding: content/test/agnostic-v3.2/00338_00.jpg (deflated 11%)
  adding: content/test/agnostic-v3.2/02283_00.jpg (deflated 14%)
  adding: content/test/agnostic-v3.2/02060_00.jpg (deflated 4%)
  adding: content/test/agnostic-v3.2/02966_00.jpg (deflated 5%)
  adding: content/test/agnostic-v3.2/02850_0

In [ ]:
!cp -r /content/test/parse/* /content/drive/MyDrive/results/agnostic-v3.2/